
Movie Recommendation System
============================
Demonstrates two classic recommendation techniques on a small sample dataset:

1. Content-Based Filtering
   - Recommends movies similar to ones a user already likes, based on
     movie attributes (genres).
   - Uses cosine similarity between genre vectors.

2. Collaborative Filtering (user-based)
   - Recommends movies liked by *other users* who have similar taste,
     based purely on the user-item rating matrix (no movie metadata needed).
   - Uses cosine similarity between user rating vectors, then predicts
     ratings for unseen movies as a weighted average of similar users' ratings.

In [5]:
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------

MOVIES = pd.DataFrame({
    "movie_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "title": [
        "The Space Between Stars",  # sci-fi, drama
        "Laugh Track",               # comedy
        "Iron Fury",                 # action, sci-fi
        "Quiet Hearts",               # romance, drama
        "Comedy Night Live",          # comedy
        "Galaxy Raiders",             # action, sci-fi
        "Tears of Yesterday",         # drama, romance
        "The Last Laugh",             # comedy, drama
    ],
    "genres": [
        ["sci-fi", "drama"],
        ["comedy"],
        ["action", "sci-fi"],
        ["romance", "drama"],
        ["comedy"],
        ["action", "sci-fi"],
        ["drama", "romance"],
        ["comedy", "drama"],
    ],
})


In [6]:
# Ratings: rows = users, columns = movie_id, 0 = not yet rated (1-5 scale)
RATINGS = pd.DataFrame(
    {
        1: [5, 1, 4, 0, 0, 5, 0, 2],   # Alice: loves sci-fi/action
        2: [1, 5, 0, 4, 5, 0, 3, 4],   # Bob: loves comedy/romance
        3: [4, 0, 5, 0, 0, 4, 0, 0],   # Carol: loves action/sci-fi
        4: [0, 4, 0, 5, 4, 0, 5, 3],   # Dave: loves drama/romance/comedy
        5: [3, 0, 4, 2, 0, 4, 0, 1],   # Eve: mixed, leans sci-fi/action
    },
    index=MOVIES["movie_id"],
).T  # users as rows, movie_id as columns

USER_NAMES = {1: "Alice", 2: "Bob", 3: "Carol", 4: "Dave", 5: "Eve"}




In [7]:
# ---------------------------------------------------------------------------
# Helper: cosine similarity
# ---------------------------------------------------------------------------

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors. Returns 0 if either is all-zero."""
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)




In [8]:
# ---------------------------------------------------------------------------
# 1. Content-based filtering
# ---------------------------------------------------------------------------

class ContentBasedRecommender:
    """Recommends movies similar to a given movie (or a user's liked movies)
    based on genre overlap."""

    def __init__(self, movies: pd.DataFrame):
        self.movies = movies.reset_index(drop=True)
        all_genres = sorted({g for genre_list in movies["genres"] for g in genre_list})
        self.genre_index = {g: i for i, g in enumerate(all_genres)}

        # Build a binary genre matrix: rows = movies, cols = genres
        matrix = np.zeros((len(self.movies), len(all_genres)))
        for i, genre_list in enumerate(self.movies["genres"]):
            for g in genre_list:
                matrix[i, self.genre_index[g]] = 1
        self.genre_matrix = matrix

    def similar_movies(self, movie_id: int, top_n: int = 3):
        """Return the top_n movies most similar in genre to the given movie."""
        idx = self.movies.index[self.movies["movie_id"] == movie_id][0]
        target_vec = self.genre_matrix[idx]

        scores = []
        for i in range(len(self.movies)):
            if i == idx:
                continue
            sim = cosine_similarity(target_vec, self.genre_matrix[i])
            scores.append((self.movies.iloc[i]["movie_id"], self.movies.iloc[i]["title"], sim))

        scores.sort(key=lambda x: x[2], reverse=True)
        return scores[:top_n]

    def recommend_for_user_profile(self, liked_movie_ids: list, top_n: int = 3):
        """Build a user 'taste profile' by averaging the genre vectors of movies
        they liked, then recommend unseen movies closest to that profile."""
        liked_idxs = self.movies.index[self.movies["movie_id"].isin(liked_movie_ids)]
        profile = self.genre_matrix[liked_idxs].mean(axis=0)

        scores = []
        for i in range(len(self.movies)):
            mid = self.movies.iloc[i]["movie_id"]
            if mid in liked_movie_ids:
                continue
            sim = cosine_similarity(profile, self.genre_matrix[i])
            scores.append((mid, self.movies.iloc[i]["title"], sim))

        scores.sort(key=lambda x: x[2], reverse=True)
        return scores[:top_n]




In [9]:
# ---------------------------------------------------------------------------
# 2. User-based collaborative filtering
# ---------------------------------------------------------------------------

class CollaborativeRecommender:
    """Recommends movies by finding users with similar rating patterns and
    predicting scores for movies the target user hasn't rated yet."""

    def __init__(self, ratings: pd.DataFrame):
        self.ratings = ratings  # rows = users, cols = movie_id, 0 = unrated

    def similar_users(self, user_id: int, top_n: int = 3):
        target = self.ratings.loc[user_id].values
        scores = []
        for other_id in self.ratings.index:
            if other_id == user_id:
                continue
            sim = cosine_similarity(target, self.ratings.loc[other_id].values)
            scores.append((other_id, sim))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_n]

    def recommend(self, user_id: int, top_n: int = 3):
        """Predict ratings for unrated movies as a similarity-weighted average
        of other users' ratings, then return the top_n highest predictions."""
        target_ratings = self.ratings.loc[user_id]
        unrated = target_ratings[target_ratings == 0].index

        similarities = {
            other_id: cosine_similarity(
                target_ratings.values, self.ratings.loc[other_id].values
            )
            for other_id in self.ratings.index
            if other_id != user_id
        }

        predictions = []
        for movie_id in unrated:
            num, denom = 0.0, 0.0
            for other_id, sim in similarities.items():
                other_rating = self.ratings.loc[other_id, movie_id]
                if other_rating > 0 and sim > 0:
                    num += sim * other_rating
                    denom += sim
            predicted = (num / denom) if denom > 0 else 0.0
            predictions.append((movie_id, predicted))

        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:top_n]




In [10]:
# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------

def title_for(movie_id: int) -> str:
    return MOVIES.loc[MOVIES["movie_id"] == movie_id, "title"].values[0]


def main():
    print("=" * 70)
    print("CONTENT-BASED FILTERING")
    print("=" * 70)
    cb = ContentBasedRecommender(MOVIES)

    seed_movie = 1
    print(f"\nBecause you watched '{title_for(seed_movie)}':")
    for movie_id, title, sim in cb.similar_movies(seed_movie, top_n=3):
        print(f"  - {title:28s}  (similarity: {sim:.2f})")

    liked = [1, 6]  # Alice's clearly sci-fi/action favorites
    print(f"\nUser taste profile built from: {[title_for(m) for m in liked]}")
    print("Recommended movies:")
    for movie_id, title, sim in cb.recommend_for_user_profile(liked, top_n=3):
        print(f"  - {title:28s}  (match score: {sim:.2f})")

    print("\n" + "=" * 70)
    print("COLLABORATIVE FILTERING (user-based)")
    print("=" * 70)
    cf = CollaborativeRecommender(RATINGS)

    for user_id in [1, 2]:
        print(f"\nUser: {USER_NAMES[user_id]}")
        print("  Most similar users:")
        for other_id, sim in cf.similar_users(user_id, top_n=2):
            print(f"    - {USER_NAMES[other_id]:8s} (similarity: {sim:.2f})")

        print("  Recommended movies (predicted rating):")
        for movie_id, predicted in cf.recommend(user_id, top_n=3):
            print(f"    - {title_for(movie_id):28s}  (predicted rating: {predicted:.2f}/5)")


if __name__ == "__main__":
    main()

CONTENT-BASED FILTERING

Because you watched 'The Space Between Stars':
  - Iron Fury                     (similarity: 0.50)
  - Quiet Hearts                  (similarity: 0.50)
  - Galaxy Raiders                (similarity: 0.50)

User taste profile built from: ['The Space Between Stars', 'Galaxy Raiders']
Recommended movies:
  - Iron Fury                     (match score: 0.87)
  - Quiet Hearts                  (match score: 0.29)
  - Tears of Yesterday            (match score: 0.29)

COLLABORATIVE FILTERING (user-based)

User: Alice
  Most similar users:
    - Carol    (similarity: 0.94)
    - Eve      (similarity: 0.93)
  Recommended movies (predicted rating):
    - Comedy Night Live             (predicted rating: 4.64/5)
    - Tears of Yesterday            (predicted rating: 3.72/5)
    - Quiet Hearts                  (predicted rating: 2.64/5)

User: Bob
  Most similar users:
    - Dave     (similarity: 0.95)
    - Eve      (similarity: 0.23)
  Recommended movies (predicted ratin